# MediQuery — Retrieval Quality Analysis

This notebook loads the JSON produced by `scripts/evaluate_mediquery.py` and visualises retrieval quality across 25 medical questions in 11 categories.

**Metrics:**
- **Keyword Recall** — fraction of expected medical terms in the answer
- **Ground-Truth Overlap** — content-word overlap with a reference answer
- **Source-Supported** — fraction of answer words grounded in retrieved passages
- **Semantic Similarity** — cosine similarity between answer and ground-truth embeddings (captures paraphrasing)

**Setup:**
```bash
pip install matplotlib numpy
uvicorn app.main:app --reload --port 8000
python scripts/evaluate_mediquery.py --out mediquery_eval_latest.json
```

In [ ]:
import json
import pathlib
import statistics

import matplotlib.pyplot as plt
import numpy as np

# ── Config ────────────────────────────────────────────────────────────────────
# Point at the JSON produced by scripts/evaluate_mediquery.py
EVAL_FILE = pathlib.Path("../mediquery_eval_20260316_142233.json")

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.family": "sans-serif",
})

with open(EVAL_FILE) as f:
    eval_data = json.load(f)

queries = eval_data["per_query"]
qs = eval_data["quality_scores"]
ls = eval_data["latency_stats"]

print(f"Loaded {eval_data['successful_queries']}/{eval_data['total_queries']} queries")
print(f"Mode:          {eval_data.get('mode', 'detailed')}")
print(f"Overall score: {eval_data['overall_score']}  Grade: {eval_data['grade']}")

## 1. Summary scorecard

In [ ]:
print("Quality metrics")
print("-" * 42)
print(f"  Keyword recall:         {qs['keyword_recall']:.3f}")
print(f"  Ground-truth overlap:   {qs['ground_truth_overlap']:.3f}")
print(f"  Source-supported:       {qs['source_supported']:.3f}")
if qs.get('semantic_similarity') is not None:
    print(f"  Semantic similarity:    {qs['semantic_similarity']:.3f}")
print(f"  Overall:                {eval_data['overall_score']:.3f}  ({eval_data['grade']})")

print("\nLatency")
print("-" * 42)
print(f"  Median:  {ls['median_ms']:.0f} ms")
print(f"  Mean:    {ls['mean_ms']:.0f} ms")
print(f"  P95:     {ls['p95_ms']:.0f} ms")
print(f"  Min:     {ls['min_ms']:.0f} ms")

## 2. Per-question scores

In [ ]:
labels = [q["question"][:45] + "…" for q in queries]
kw  = [q["scores"]["keyword_recall"] for q in queries]
gt  = [q["scores"]["ground_truth_overlap"] for q in queries]
src = [q["scores"]["source_supported"] for q in queries]
sem = [q["scores"].get("semantic_similarity") for q in queries]
has_sem = any(s is not None for s in sem)

x = np.arange(len(labels))
n_bars = 4 if has_sem else 3
w = 0.8 / n_bars

fig, ax = plt.subplots(figsize=(17, 7))
ax.bar(x - w * 1.5, kw,  w, label="Keyword recall",       color="#2563eb", alpha=0.85)
ax.bar(x - w * 0.5, gt,  w, label="Ground-truth overlap", color="#16a34a", alpha=0.85)
ax.bar(x + w * 0.5, src, w, label="Source-supported",     color="#dc2626", alpha=0.85)
if has_sem:
    sem_vals = [s if s is not None else 0 for s in sem]
    ax.bar(x + w * 1.5, sem_vals, w, label="Semantic similarity", color="#7c3aed", alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Score (0–1)")
ax.set_title("Per-question quality scores", fontsize=13, fontweight="bold")
ax.set_ylim(0, 1.05)
ax.legend()
plt.tight_layout()
plt.savefig("per_question_scores.png", dpi=150)
plt.show()

## 3. Per-category breakdown

In [ ]:
by_cat = {}
for q in queries:
    by_cat.setdefault(q.get("category", "unknown"), []).append(q)

cats = sorted(by_cat)
cat_kw  = [statistics.mean(r["scores"]["keyword_recall"] for r in by_cat[c]) for c in cats]
cat_gt  = [statistics.mean(r["scores"]["ground_truth_overlap"] for r in by_cat[c]) for c in cats]
cat_src = [statistics.mean(r["scores"]["source_supported"] for r in by_cat[c]) for c in cats]
cat_sem_raw = [
    [r["scores"].get("semantic_similarity") for r in by_cat[c] if r["scores"].get("semantic_similarity") is not None]
    for c in cats
]
cat_sem = [statistics.mean(v) if v else None for v in cat_sem_raw]

x = np.arange(len(cats))
w = 0.2

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x - w * 1.5, cat_kw,  w, label="Keyword recall",       color="#2563eb", alpha=0.85)
ax.bar(x - w * 0.5, cat_gt,  w, label="Ground-truth overlap", color="#16a34a", alpha=0.85)
ax.bar(x + w * 0.5, cat_src, w, label="Source-supported",     color="#dc2626", alpha=0.85)
if any(s is not None for s in cat_sem):
    cat_sem_vals = [s if s is not None else 0 for s in cat_sem]
    ax.bar(x + w * 1.5, cat_sem_vals, w, label="Semantic similarity", color="#7c3aed", alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels([c.replace("_", " ") for c in cats], rotation=30, ha="right", fontsize=9)
ax.set_ylabel("Score (0–1)")
ax.set_title("Average scores by medical category", fontsize=13, fontweight="bold")
ax.set_ylim(0, 1.05)
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig("category_scores.png", dpi=150)
plt.show()

## 4. Latency distribution

In [ ]:
latencies = [q["latency_ms"] for q in queries]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(latencies, bins=10, color="#6c47ff", alpha=0.8, edgecolor="white")
axes[0].axvline(statistics.median(latencies), color="#dc2626", linestyle="--",
                label=f"Median {statistics.median(latencies):.0f} ms")
axes[0].axvline(ls["p95_ms"], color="#ea580c", linestyle=":",
                label=f"P95 {ls['p95_ms']:.0f} ms")
axes[0].set_xlabel("Latency (ms)")
axes[0].set_ylabel("Count")
axes[0].set_title("Latency distribution")
axes[0].legend()

axes[1].scatter(range(len(latencies)), latencies, color="#6c47ff", alpha=0.7, s=40)
axes[1].axhline(statistics.median(latencies), color="#dc2626", linestyle="--", linewidth=1)
axes[1].set_xlabel("Question index")
axes[1].set_ylabel("Latency (ms)")
axes[1].set_title("Latency per query")

plt.tight_layout()
plt.savefig("latency_distribution.png", dpi=150)
plt.show()

## 5. Retrieval confidence scores

In [ ]:
top_scores = []
for q in queries:
    scores = [s["score"] for s in q["sources"] if s.get("score") is not None]
    if scores:
        top_scores.append(max(scores))

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(top_scores, bins=12, color="#16a34a", alpha=0.8, edgecolor="white")
ax.axvline(0.35, color="#dc2626", linestyle="--", label="MIN_SCORE threshold (0.35)")
ax.set_xlabel("Top Pinecone similarity score")
ax.set_ylabel("Count")
ax.set_title("Distribution of top retrieval confidence scores", fontsize=12)
ax.legend()
plt.tight_layout()
plt.savefig("retrieval_confidence.png", dpi=150)
plt.show()

if top_scores:
    print(f"Mean top score:   {statistics.mean(top_scores):.3f}")
    print(f"Median top score: {statistics.median(top_scores):.3f}")
    print(f"Queries below 0.35 threshold: {sum(1 for s in top_scores if s < 0.35)}")

## 6. Lexical vs semantic quality (if semantic enabled)

In [ ]:
sem_vals = [q["scores"].get("semantic_similarity") for q in queries]
gt_vals  = [q["scores"]["ground_truth_overlap"] for q in queries]

if any(s is not None for s in sem_vals):
    paired = [(gt, sem) for gt, sem in zip(gt_vals, sem_vals) if sem is not None]
    x_gt, y_sem = zip(*paired)

    fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(x_gt, y_sem, color="#7c3aed", alpha=0.7, s=60)
    ax.set_xlabel("Ground-truth word overlap")
    ax.set_ylabel("Semantic similarity (cosine)")
    ax.set_title("Lexical overlap vs semantic similarity", fontsize=12)

    # Correlation
    if len(paired) > 2:
        n = len(paired)
        mean_x, mean_y = sum(x_gt) / n, sum(y_sem) / n
        num = sum((x - mean_x) * (y - mean_y) for x, y in paired)
        den = (sum((x - mean_x)**2 for x in x_gt) * sum((y - mean_y)**2 for y in y_sem)) ** 0.5
        r = num / den if den else 0
        ax.set_title(f"Lexical overlap vs semantic similarity  (r = {r:.2f})", fontsize=12)

    plt.tight_layout()
    plt.savefig("lexical_vs_semantic.png", dpi=150)
    plt.show()
else:
    print("Semantic similarity not available in this eval file.")
    print("Re-run with HF_API_TOKEN set to enable it.")

## 7. Worst-performing questions

In [ ]:
def avg_score(q):
    vals = [q["scores"][k] for k in ("keyword_recall", "ground_truth_overlap", "source_supported")]
    sem = q["scores"].get("semantic_similarity")
    if sem is not None:
        vals.append(sem)
    return sum(vals) / len(vals)

worst = sorted(queries, key=avg_score)[:5]

for i, q in enumerate(worst, 1):
    avg = avg_score(q)
    s = q["scores"]
    sem_str = f"  sem={s['semantic_similarity']}" if s.get('semantic_similarity') is not None else ""
    print(f"{i}. [{avg:.2f}] [{q.get('category', '?')}] {q['question']}")
    print(f"   kw={s['keyword_recall']}  gt={s['ground_truth_overlap']}  src={s['source_supported']}{sem_str}")
    print()

## 8. Improvement roadmap

| Priority | Change | Expected impact |
|---|---|---|
| 🔴 High | Switch to `pritamdeka/S-PubMedBert-MS-MARCO` (768-dim biomedical) | +10–15 pts keyword recall |
| 🔴 High | Rebuild Pinecone index at 768-dim with new embeddings | Required for above |
| 🟢 Done | Cross-encoder reranker (`cross-encoder/ms-marco-MiniLM-L-6-v2`) | +5–8 pts source-supported |
| 🟢 Done | MIN_SCORE=0.35 threshold — drop low-relevance chunks | Fewer hallucinations |
| 🟢 Done | Brief/Detailed mode toggle | UX improvement |
| 🟢 Done | Semantic similarity metric in evaluation | Better quality signal |